In [4]:
import pandas as pd


In [5]:
df = pd.read_csv('gk_qna_dataset.csv')

In [6]:
df['question'][0]

'What is the capital of "France"?'

###### use 'regular expression website' to clean \"' from string

##### Tokenization

In [7]:
import re
def tokenize(text):
    
    # lowercase
    text = text.lower()
    # remove quotes
    text = re.sub(r" [\"'] ","",text)
    # remove all punctuatuations
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    # remove extra spaces
    text = re.sub(r"\s+"," ",text.strip())
    # tokenize (split)
    tokens = text.split()
    return tokens

In [8]:
print(tokenize(df['question'][0]))

['what', 'is', 'the', 'capital', 'of', 'france']


##### Vocab forming

In [9]:
vocab = {'<UNK>':0}

def build_vocab(row):
    # print(row['question'],row['answer']) 
    
    tokenize_question = tokenize(row['question'])
    tokenize_answer = tokenize(row['answer'])
    
    # print(tokenize_question, tokenize_answer)
    
    merged_tokens = tokenize_question + tokenize_answer  # concatanation
    # print(merged_tokens)
    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

In [10]:
df.apply(build_vocab, axis=1)

0      None
1      None
2      None
3      None
4      None
       ... 
166    None
167    None
168    None
169    None
170    None
Length: 171, dtype: object

In [11]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 'iraq

##### text to indices

In [12]:
def text_to_indices(text, vocab):
    indexed_text = []
    
    for token in tokenize(text):
        # print(token)
        if token in vocab:
            # print(vocab[token])
            indexed_text.append(vocab[token])

        else:
            indexed_text.append(vocab['<UNK>'])
            
    return indexed_text

text_to_indices('phitron',vocab)

[0]

In [15]:
index = 0
df.iloc[index]

question    What is the capital of "France"?
answer                                 Paris
Name: 0, dtype: object

##### Dataset and Dataloader

In [18]:
import torch
from torch.utils.data import Dataset,DataLoader

class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df = df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)
        
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [20]:
dataset = QADataset(df,vocab)
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [25]:
dataloader = DataLoader(dataset , batch_size=1, shuffle=True)

In [31]:
#use of squeeze function 
x = torch.tensor( [[[1,2,3]]])
print(x.shape)

y = x.squeeze(0)
print(y.shape)

torch.Size([1, 1, 3])
torch.Size([1, 3])


##### RNN architecture implementation

In [39]:
import torch.nn as nn

class simpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim , hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, question):
        embedded = self.embedding(question)
        _, final = self.rnn(embedded)
        
        return self.fc(final.squeeze(0))

In [40]:
model = simpleRNN(vocab_size=len(vocab))

##### Training loop

In [41]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
epochs = 50

In [42]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        output = model(question)
        # Fix: take only the first answer token as target → shape (1,)
        target = answer[0][0].unsqueeze(0)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  |  Loss: {total_loss:.4f}")


Epoch  10/50  |  Loss: 594.4364
Epoch  20/50  |  Loss: 369.0134
Epoch  30/50  |  Loss: 246.8651
Epoch  40/50  |  Loss: 174.2716
Epoch  50/50  |  Loss: 122.8851


In [43]:
vocab.items()

dict_items([('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9), ('italy', 10), ('rome', 11), ('spain', 12), ('madrid', 13), ('japan', 14), ('tokyo', 15), ('canada', 16), ('ottawa', 17), ('brazil', 18), ('brasilia', 19), ('australia', 20), ('canberra', 21), ('india', 22), ('new', 23), ('delhi', 24), ('china', 25), ('beijing', 26), ('russia', 27), ('moscow', 28), ('united', 29), ('states', 30), ('washington', 31), ('dc', 32), ('mexico', 33), ('city', 34), ('egypt', 35), ('cairo', 36), ('turkey', 37), ('ankara', 38), ('argentina', 39), ('buenos', 40), ('aires', 41), ('south', 42), ('korea', 43), ('seoul', 44), ('indonesia', 45), ('jakarta', 46), ('pakistan', 47), ('islamabad', 48), ('bangladesh', 49), ('dhaka', 50), ('nepal', 51), ('kathmandu', 52), ('sri', 53), ('lanka', 54), ('colombo', 55), ('thailand', 56), ('bangkok', 57), ('malaysia', 58), ('kuala', 59), ('lumpur', 60), ('vietnam', 61), ('hanoi', 62

In [44]:
idx_to_word = { idx : word for word,idx in vocab.items()  }
idx_to_word

{0: '<UNK>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france',
 7: 'paris',
 8: 'germany',
 9: 'berlin',
 10: 'italy',
 11: 'rome',
 12: 'spain',
 13: 'madrid',
 14: 'japan',
 15: 'tokyo',
 16: 'canada',
 17: 'ottawa',
 18: 'brazil',
 19: 'brasilia',
 20: 'australia',
 21: 'canberra',
 22: 'india',
 23: 'new',
 24: 'delhi',
 25: 'china',
 26: 'beijing',
 27: 'russia',
 28: 'moscow',
 29: 'united',
 30: 'states',
 31: 'washington',
 32: 'dc',
 33: 'mexico',
 34: 'city',
 35: 'egypt',
 36: 'cairo',
 37: 'turkey',
 38: 'ankara',
 39: 'argentina',
 40: 'buenos',
 41: 'aires',
 42: 'south',
 43: 'korea',
 44: 'seoul',
 45: 'indonesia',
 46: 'jakarta',
 47: 'pakistan',
 48: 'islamabad',
 49: 'bangladesh',
 50: 'dhaka',
 51: 'nepal',
 52: 'kathmandu',
 53: 'sri',
 54: 'lanka',
 55: 'colombo',
 56: 'thailand',
 57: 'bangkok',
 58: 'malaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'vietnam',
 62: 'hanoi',
 63: 'uae',
 64: 'abu',
 65: 'dhabi',
 66: 'iran',
 67: 'tehran',
 68: '

In [45]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indices(question_text,vocab)

    if not indices:
      return "<UNK>"

    x = torch.tensor(indices).unsqueeze(0)  #batch size, sequence legth

    pred = torch.argmax(model(x),dim=1).item()


  return idx_to_word.get(pred,"<UNK>")

In [46]:
# Test
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which planet is known as the red planet?",
    "Which bird cannot fly?",
    "What is the capital of Japan ?",

]
for q in tests:
    print(f"{q:45} → {predict(q, model, vocab, idx_to_word)}")

What is the capital of France?                → paris
What is H2O commonly called?                  → water
Which planet is known as the red planet?      → mars
Which bird cannot fly?                        → ostrich
What is the capital of Japan ?                → tokyo
